In [13]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import func
from sqlalchemy.orm import Query

import src
from src.data.models import Channel
from src.data.models import Comment
from src.data.models import Video

In [33]:
engine = create_engine(src.PS_ENGINE)
pd.set_option("display.max_rows", 1024)
pd.set_option("display.max_colwidth", 1024)

In [75]:
base_query = (
    Query(Channel)
    .join(Video)
    .join(Comment)
    .group_by(Channel)
    .filter(Video.is_valid == True, Comment.is_valid == True)
)

total_comments = base_query.with_entities(
    Channel.channel, func.count(Comment.id).label("total_comments"),
)

root_comments = base_query.filter(Comment.parent == "root").with_entities(
    Channel.channel, func.count(Comment.id).label("root_comments"),
)

reaction_comments = base_query.filter(
    Comment.author_is_uploader == True, Comment.parent != "root",
).with_entities(Channel.channel, func.count(Comment.id).label("reaction_comments"))

n_videos = (
    Query(Channel)
    .join(Video)
    .filter(Video.is_valid == True)
    .group_by(Channel)
    .with_entities(Channel.channel, func.count(Video.id).label("n_videos"))
)

n_different_commenters = base_query.with_entities(
    Channel.channel, func.count(Comment.author.distinct()).label("distinct_commenters"),
)

with engine.connect() as conn:
    df = pd.read_sql(total_comments.statement, conn)
    root_df = pd.read_sql(root_comments.statement, conn)
    reaction_df = pd.read_sql(reaction_comments.statement, conn)
    n_commenters = pd.read_sql(n_different_commenters.statement, conn)

    n_videos_df = pd.read_sql(n_videos.statement, conn)

    for d in [root_df, reaction_df, n_videos_df, n_commenters]:
        df = pd.merge(df, d, on="channel", how="outer")

    df["avg_comments_per_vid"] = df["total_comments"] / df["n_videos"]
    df["avg_reaction_comments_per_vid"] = df["reaction_comments"] / df["n_videos"]
    df["avg_root_comments_per_vid"] = df["root_comments"] / df["n_videos"]

In [78]:
n_comments_per_commenter = base_query.group_by(Comment.author).with_entities(
    Channel.channel, Comment.author, func.count(Comment.id),
)

with engine.connect() as conn:
    df = pd.read_sql(n_comments_per_commenter.statement, conn)

In [88]:
df.groupby("channel").describe()

count_1                                            \
                           count       mean        std  min  25%  50%  75%   
channel                                                                      
AfD TV                  100155.0   5.823833  21.300802  1.0  1.0  2.0  4.0   
AfD-Fraktion Bundestag  194784.0  10.652045  44.968426  1.0  1.0  2.0  6.0   
BÜNDNIS 90/DIE GRÜNEN       48.0   1.625000   1.248403  1.0  1.0  1.0  2.0   
CDU                      10426.0   2.628717   9.049100  1.0  1.0  1.0  2.0   
CSU                        484.0   2.237603   3.516208  1.0  1.0  1.0  2.0   
DIE LINKE                 5174.0   3.966564  17.117571  1.0  1.0  1.0  2.0   
FDP                        302.0   1.092715   0.421215  1.0  1.0  1.0  1.0   
SPD                       6885.0   1.791140  10.202309  1.0  1.0  1.0  1.0   

                                
                           max  
channel                         
AfD TV                  2813.0  
AfD-Fraktion Bundestag  5259.0  
BÜNDNIS 90/DIE GRÜNEN      7.0  
CDU                      404.0  
CSU                       37.0  
DIE LINKE                641.0  
FDP                        6.0  
SPD                      747.0

In [77]:
df.sort_values("reaction_comments", ascending=False).fillna(0).T

,7,5,0,1,6,2,3,4
channel,SPD,DIE LINKE,AfD TV,AfD-Fraktion Bundestag,FDP,BÜNDNIS 90/DIE GRÜNEN,CDU,CSU
total_comments,12332,20523,583286,2074848,330,78,27407,1083
root_comments,7870,10859,376690,1464653,309,45,16807,898
reaction_comments,715.0,143.0,134.0,17.0,2.0,0.0,0.0,0.0
n_videos,493,456,1463,5209,502,460,666,145
distinct_commenters,6885,5174,100154,194783,302,48,10426,484
avg_comments_per_vid,25.014199,45.006579,398.691729,398.319831,0.657371,0.169565,41.151652,7.468966
avg_reaction_comments_per_vid,1.450304,0.313596,0.091593,0.003264,0.003984,0.0,0.0,0.0
avg_root_comments_per_vid,15.963489,23.813596,257.477785,281.177385,0.615538,0.097826,25.235736,6.193103
